# Train Log-Mel + CNN-GRU trên Google Colab

Notebook này tương đương pipeline trong `src/training/train.py`: tải Speech Commands, trích xuất Log-Mel, train CNN-GRU, chọn checkpoint theo validation macro-F1 và đánh giá trên test set.

Trước khi chạy, bật GPU tại **Runtime > Change runtime type > T4 GPU** và đưa toàn bộ thư mục project lên Colab hoặc Google Drive. Không cần upload `data/raw` và `outputs`.

In [ ]:
# Chỉ bỏ comment hai dòng dưới nếu project nằm trong Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')

from pathlib import Path
import os

# Sửa đường dẫn này cho đúng vị trí project trên Colab.
PROJECT_DIR = Path('/content/SPPR-Speech-Command-Recognition-for-Robot-Navigation')
assert (PROJECT_DIR / 'src').is_dir(), f'Không tìm thấy project tại: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())

## Cài dependency cho training

Colab thường đã có `torch` và `torchaudio` tương thích với GPU, vì vậy cell này chỉ cài các gói còn lại và không thay thế bản PyTorch của runtime.

In [ ]:
!pip install -q PyYAML tqdm scikit-learn matplotlib soundfile

In [ ]:
import torch
from torch import nn

from src.data.dataset import create_datasets
from src.features.logmel import build_logmel_extractor
from src.models.cnn_gru import build_model
from src.training.evaluate import evaluate_checkpoint, evaluate_model
from src.training.train import build_loader, save_checkpoint, train_one_epoch
from src.utils.config import load_config
from src.utils.metrics import compute_classification_metrics
from src.utils.seed import resolve_device, set_seed

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CẢNH BÁO: Runtime đang dùng CPU. Hãy bật T4 GPU rồi chạy lại từ đầu.')

## Đọc cấu hình và chuẩn bị dữ liệu

Lần chạy đầu tiên sẽ tự tải Speech Commands. Có thể đổi `EPOCHS` hoặc `CHECKPOINT_PATH`. Nếu muốn checkpoint tồn tại khi Colab ngắt kết nối, hãy mount Drive và đặt `CHECKPOINT_PATH` vào thư mục Drive.

In [ ]:
CONFIG_PATH = 'configs/cnn_gru.yaml'
EPOCHS = 40
CHECKPOINT_PATH = PROJECT_DIR / 'outputs/checkpoints/best_cnn_gru.pt'
# Ví dụ lưu bền vững trên Drive:
# CHECKPOINT_PATH = Path('/content/drive/MyDrive/cnn_gru/best_cnn_gru.pt')

config = load_config(CONFIG_PATH)
DATA_ROOT = Path('/content/data/raw')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
config['data']['root'] = str(DATA_ROOT)
config['training']['epochs'] = EPOCHS
config['training']['checkpoint_path'] = str(CHECKPOINT_PATH)
set_seed(config['seed'])
device = resolve_device(config.get('device', 'auto'))

train_dataset, val_dataset, _ = create_datasets(config)
train_loader = build_loader(train_dataset, config, shuffle=True, device=device)
val_loader = build_loader(val_dataset, config, shuffle=False, device=device)

print('Device:', device)
print('Classes:', train_dataset.classes)
print('Train samples:', len(train_dataset))
print('Validation samples:', len(val_dataset))

## Khởi tạo CNN-GRU

In [ ]:
model = build_model(config, num_classes=len(train_dataset.classes)).to(device)
feature_extractor = build_logmel_extractor(config).to(device)
criterion = nn.CrossEntropyLoss()
train_cfg = config['training']
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=train_cfg['learning_rate'],
    weight_decay=train_cfg.get('weight_decay', 0.0),
)

parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {parameter_count:,}')
print(model)

## Train và lưu model tốt nhất

Checkpoint được chọn theo validation macro-F1; validation loss dùng để phân xử khi macro-F1 bằng nhau.

In [ ]:
best_val_macro_f1 = -1.0
best_val_loss = float('inf')
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy = train_one_epoch(
        model=model,
        feature_extractor=feature_extractor,
        data_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        epoch=epoch,
        gradient_clip_norm=train_cfg.get('gradient_clip_norm'),
    )
    val_loss, val_accuracy, y_true, y_pred = evaluate_model(
        model=model,
        feature_extractor=feature_extractor,
        data_loader=val_loader,
        criterion=criterion,
        device=device,
        show_progress=False,
    )
    val_metrics, _ = compute_classification_metrics(
        y_true, y_pred, train_dataset.classes
    )
    val_macro_f1 = val_metrics['macro_f1']
    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'train_accuracy': train_accuracy,
        'val_loss': val_loss,
        'val_accuracy': val_accuracy,
        'val_macro_f1': val_macro_f1,
    })

    print(
        f'epoch={epoch:02d} '
        f'train_loss={train_loss:.4f} train_acc={train_accuracy:.4f} '
        f'val_loss={val_loss:.4f} val_acc={val_accuracy:.4f} '
        f'val_macro_f1={val_macro_f1:.4f}'
    )

    is_best = val_macro_f1 > best_val_macro_f1 or (
        val_macro_f1 == best_val_macro_f1 and val_loss < best_val_loss
    )
    if is_best:
        best_val_macro_f1 = val_macro_f1
        best_val_loss = val_loss
        save_checkpoint(
            path=str(CHECKPOINT_PATH),
            model=model,
            optimizer=optimizer,
            config=config,
            classes=train_dataset.classes,
            epoch=epoch,
            val_loss=val_loss,
            val_accuracy=val_accuracy,
            val_macro_f1=val_macro_f1,
        )
        print('  -> Saved best checkpoint:', CHECKPOINT_PATH)

print(f'Best validation macro-F1: {best_val_macro_f1:.4f}')

## Đánh giá checkpoint tốt nhất trên test set

In [ ]:
test_metrics = evaluate_checkpoint(
    config=config,
    checkpoint_path=str(CHECKPOINT_PATH),
    split='testing',
    prefix='test',
)
print(f"Test accuracy: {test_metrics['accuracy']:.4f}")
print(f"Test macro-F1: {test_metrics['macro_f1']:.4f}")
print('Per-class metrics:')
for class_name, metrics in test_metrics['per_class'].items():
    print(
        f"  {class_name:>8}: precision={metrics['precision']:.4f} "
        f"recall={metrics['recall']:.4f} f1={metrics['f1']:.4f}"
    )

## Tải checkpoint `.pt` về máy

In [ ]:
from google.colab import files

assert CHECKPOINT_PATH.is_file(), f'Không tìm thấy checkpoint: {CHECKPOINT_PATH}'
files.download(str(CHECKPOINT_PATH))